In [38]:
#jupyter nbconvert --to python preprocesamiento.ipynb. #Convertir codigo python punta a punta

import pandas as pd
import numpy as np

#LOAD DATASETS
datafiles = {
    "Fenomenos": "../Dataset_INUMET/Fenomenos.csv",
    "Humedad": "../Dataset_INUMET/Humedad.csv",
    "Lluvia": "../Dataset_INUMET/Lluvia.csv",
    "Temperatura": "../Dataset_INUMET/Temperatura.csv",
    "Visibilidad": "../Dataset_INUMET/Visibilidad.csv",
    "VientoSalto": "../Dataset_INUMET/VientoSalto.csv",
    "VientoCarrasco": "../Dataset_INUMET/VientoCarrasco.csv"
}

dataframes = {}
for name, path in datafiles.items():
    dataframes[name] = pd.read_csv(path, sep = ";", encoding='utf-8')
# Access data like: dataframes["Temperature"]


In [39]:
# ====== 1. Lluvia.csv ======
print("Procesando datos Lluvia.csv")
df_lluvia = dataframes["Lluvia"].copy()

df_lluvia = df_lluvia.rename(columns={
    "Estación": "estacion",
    "Cod. Pluvio": "cod_pluvio",
    "Fecha": "fecha",
    "[mm]": "precip_mm",
    "Comentario": "coment_lluv"
})

# Convertir fecha y precipitación
df_lluvia["fecha"] = pd.to_datetime(df_lluvia["fecha"], format="%d/%m/%Y", errors="coerce")
df_lluvia["precip_mm"] = df_lluvia["precip_mm"].replace("TRAZA", "0.0")  #! Nota: Transformación TRAZA se traduce a 0.0 
df_lluvia["precip_mm"] = df_lluvia["precip_mm"].astype(float)

#! Criterio: día lluvioso si precipitación >= 1.0 mm
precip_minima = 1.0
df_lluvia["dia_lluvioso"] = (df_lluvia["precip_mm"] >= precip_minima).astype(int)

# Seleccionamos solo columnas útiles para el df_global
df_lluvia = df_lluvia[["fecha", "precip_mm", "dia_lluvioso"]]

print(df_lluvia.head())


Procesando datos Lluvia.csv
       fecha  precip_mm  dia_lluvioso
0 1950-01-01        4.6             1
1 1950-01-02        0.0             0
2 1950-01-03        0.0             0
3 1950-01-04        0.0             0
4 1950-01-05        0.0             0


In [40]:
# ====== 2. Humedad.csv ======
print("Procesando datos Humedad.csv")
df_hum = dataframes["Humedad"].copy()

df_hum = df_hum.rename(columns={
    "Fecha/hora": "fecha",
    "HumedadCarrasco ": "hum_carrasco",  
    "HumedadSalto ": "hum_salto"
})

# Convertir fecha
df_hum["fecha"] = pd.to_datetime(df_hum["fecha"], errors="coerce")
df_hum["fecha"] = df_hum["fecha"].dt.date  # Extraer solo la fecha (sin hora)
df_hum["fecha"] = pd.to_datetime(df_hum["fecha"])  # Volver a datetime

# Eliminar columnas no deseadas
df_hum = df_hum.drop(columns=["Unnamed: 3", "Unnamed: 4"], errors="ignore")  # errors="ignore" evita error si no existen

# Agrupar por fecha (en caso de múltiples mediciones por día)
df_hum = df_hum.groupby("fecha").agg({
    "hum_carrasco": "mean",
    "hum_salto": "mean"
}).reset_index()

print(df_hum.head())
print(df_hum.isna().sum())

Procesando datos Humedad.csv
       fecha  hum_carrasco  hum_salto
0 1980-01-01           NaN       69.0
1 1980-01-02          60.0       61.0
2 1980-01-03          76.0       63.0
3 1980-01-04          60.0       82.0
4 1980-01-05          66.0       84.0
fecha              0
hum_carrasco      12
hum_salto       1425
dtype: int64


In [41]:
# ====== 3. Temperatura.csv PERCENTILES para tmax y tmin ======
print("Procesando datos Temperatura.csv")

df_temp = dataframes["Temperatura"].copy()

df_temp = df_temp.rename(columns={
    "Fecha": "fecha",
    "TmáxCarrasco": "tmax_carrasco",
    "TmínCarrasco": "tmin_carrasco",
    "TmáxSalto": "tmax_salto",
    "TmínSalto": "tmin_salto"
})

df_temp["fecha"] = pd.to_datetime(df_temp["fecha"], errors="coerce", dayfirst=True) 
# Eliminar fechas no válidas
df_temp = df_temp.dropna(subset=["fecha"]) #! existen varias fechas no definidas, nAT 
df_temp = df_temp.drop(columns=["Unnamed: 5"], errors="ignore")  

df_temp["mes"] = df_temp["fecha"].dt.month

# emperatura media por mes (para cada columna)
media_por_mes = df_temp.groupby("mes")[[
    "tmax_carrasco", 
    "tmin_carrasco", 
    "tmax_salto", 
    "tmin_salto"
]].mean()
#print(df_temp[["tmax_carrasco", "tmin_carrasco", "tmax_salto", "tmin_salto"]].isna().sum())
# llenar NaN con la media del mes, para cada colmn
for col in ["tmax_carrasco", "tmin_carrasco", "tmax_salto", "tmin_salto"]:
    # Mapear el valor del mes a la media correspondiente
    media_col = media_por_mes[col]
    df_temp[col] = df_temp[col].fillna(df_temp["mes"].map(media_col))


df_temp["amplitud_carrasco"] = df_temp["tmax_carrasco"] - df_temp["tmin_carrasco"]
df_temp["amplitud_salto"] = df_temp["tmax_salto"] - df_temp["tmin_salto"]

# Calcular percentiles clave
p10_maxc, p30_maxc, p70_maxc, p90_maxc = np.percentile(df_temp["tmax_carrasco"], [10, 30, 70, 90])
p10_minc, p30_minc, p70_minc, p90_minc = np.percentile(df_temp["tmin_carrasco"], [10, 30, 70, 90])

p10_maxs, p30_maxs, p70_maxs, p90_maxs = np.percentile(df_temp["tmax_salto"], [10, 30, 70, 90])
p10_mins, p30_mins, p70_mins, p90_mins = np.percentile(df_temp["tmin_salto"], [10, 30, 70, 90])

# Mostrar umbrales
print(f"Carrasco MAX - Percentiles: P10={p10_maxc:.1f}°C, P30={p30_maxc:.1f}°C, P70={p70_maxc:.1f}°C, P90={p90_maxc:.1f}°C")
print(f"Carrasco MIN - Percentiles: P10={p10_minc:.1f}°C, P30={p30_minc:.1f}°C, P70={p70_minc:.1f}°C, P90={p90_minc:.1f}°C")

print(f"Salto MAX - Percentiles: P10={p10_maxs:.1f}°C, P30={p30_maxs:.1f}°C, P70={p70_maxs:.1f}°C, P90={p90_maxs:.1f}°C")
print(f"Salto  MIN  - Percentiles: P10={p10_mins:.1f}°C, P30={p30_mins:.1f}°C, P70={p70_mins:.1f}°C, P90={p90_mins:.1f}°C")
# Función de categorización basada en percentiles
def categorizar_por_percentiles_tmin_tmax(temp, p10, p30, p70, p90):
    if temp < p10:
        return "muy_frio"
    elif temp < p30:
        return "fresco"
    elif temp < p70:
        return "agradable"
    elif temp < p90:
        return "calido"
    else:
        return "muy_calido"
    
def categorizar_por_percentiles_amplitud(temp, p25, p50, p85):
    if temp < p25:
        return "variacion_baja"
    elif temp < p50:
        return "variacion_moderada"
    elif temp < p85:
        return "variacion_alta"
    else:
        return "variacion_muy_alta"


# Aplicar discretización
df_temp["tmax_carrasco"] = df_temp["tmax_carrasco"].apply(
    lambda t: categorizar_por_percentiles_tmin_tmax(t, p10_maxc, p30_maxc, p70_maxc, p90_maxc)
)
df_temp["tmin_carrasco"] = df_temp["tmin_carrasco"].apply(
    lambda t: categorizar_por_percentiles_tmin_tmax(t, p10_minc, p30_minc, p70_minc, p90_minc)
)
df_temp["tmax_salto"] = df_temp["tmax_salto"].apply(
    lambda t: categorizar_por_percentiles_tmin_tmax(t, p10_maxc, p30_maxc, p70_maxc, p90_maxc)
)
df_temp["tmin_salto"] = df_temp["tmin_salto"].apply(
    lambda t: categorizar_por_percentiles_tmin_tmax(t, p10_minc, p30_minc, p70_minc, p90_minc)
)

# ====== Discretización de amplitud térmica ======

# Calcular percentiles para amplitud
p25_amp_c = np.percentile(df_temp["amplitud_carrasco"], 25)
p65_amp_c = np.percentile(df_temp["amplitud_carrasco"], 50)
p85_amp_c = np.percentile(df_temp["amplitud_carrasco"], 85)

p25_amp_s = np.percentile(df_temp["amplitud_salto"], 25)
p65_amp_s = np.percentile(df_temp["amplitud_salto"], 50)
p85_amp_s = np.percentile(df_temp["amplitud_salto"], 85)

# Mostrar umbrales (opcional)
# print(f"\nCarrasco - Amplitud: P25={p25_amp_c:.1f}°C, P50={p65_amp_c:.1f}°C, P85={p85_amp_c:.1f}°C")
# print(f"Salto    - Amplitud: P25={p25_amp_s:.1f}°C, P50={p65_amp_s:.1f}°C, P85={p85_amp_s:.1f}°C")

# Aplicar categorización a amplitud
df_temp["amplitud_carrasco"] = df_temp["amplitud_carrasco"].apply(
    lambda t: categorizar_por_percentiles_amplitud(t, p25_amp_c, p65_amp_c, p85_amp_c)
)

df_temp["amplitud_salto"] = df_temp["amplitud_salto"].apply(
    lambda t: categorizar_por_percentiles_amplitud(t, p25_amp_s, p65_amp_s, p85_amp_s)
)

# Seleccionar solo columnas finales
df_temp = df_temp[[
    "fecha",
    "tmax_carrasco",
    "tmin_carrasco",
    "tmax_salto",  
    "tmin_salto",  
    "amplitud_carrasco",
    "amplitud_salto"  
]]

# Mostrar resultados
df_temp

Procesando datos Temperatura.csv
Carrasco MAX - Percentiles: P10=13.1°C, P30=17.4°C, P70=24.5°C, P90=29.0°C
Carrasco MIN - Percentiles: P10=5.0°C, P30=9.4°C, P70=15.7°C, P90=19.2°C
Salto MAX - Percentiles: P10=16.0°C, P30=21.2°C, P70=29.0°C, P90=33.5°C
Salto  MIN  - Percentiles: P10=4.4°C, P30=9.8°C, P70=16.8°C, P90=20.4°C


,fecha,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto
0,1970-01-01,muy_calido,muy_calido,muy_calido,muy_calido,variacion_alta,variacion_moderada
1,1970-01-02,muy_calido,muy_calido,muy_calido,muy_calido,variacion_alta,variacion_alta
2,1970-01-03,calido,muy_calido,muy_calido,muy_calido,variacion_moderada,variacion_moderada
3,1970-01-04,agradable,calido,muy_calido,muy_calido,variacion_baja,variacion_alta
4,1970-01-05,muy_calido,muy_calido,muy_calido,muy_calido,variacion_alta,variacion_alta
...,...,...,...,...,...,...,...
20303,2025-08-03,fresco,fresco,agradable,agradable,variacion_alta,variacion_moderada
20304,2025-08-04,fresco,fresco,agradable,fresco,variacion_moderada,variacion_moderada
20305,2025-08-05,muy_frio,fresco,fresco,muy_frio,variacion_baja,variacion_alta
20306,2025-08-06,fresco,muy_frio,agradable,muy_frio,variacion_alta,variacion_alta


In [47]:
# Merge de dataset procesados

print("Combinando datasets en df_global")

df_global = df_lluvia.copy()
# Hacer merge con humedad
df_global = pd.merge(df_global, df_hum, on="fecha", how="outer")

# Hacer merge con temperatura
#df_global = pd.merge(df_global, df_temp, on="fecha", how="outer")
df_global = pd.merge(df_global, df_temp, on="fecha", how="outer")

df_global = df_global[[ "fecha",  
                       "hum_carrasco", "hum_salto",
                       "tmax_carrasco", "tmin_carrasco", "tmax_salto", "tmin_salto", "amplitud_carrasco", "amplitud_salto", "dia_lluvioso"]]

print(df_global)
df_global.to_csv('../Dataset_INUMET/data.csv', index=False)

Combinando datasets en df_global
           fecha  hum_carrasco  hum_salto tmax_carrasco tmin_carrasco  \
0     1950-01-01           NaN        NaN           NaN           NaN   
1     1950-01-02           NaN        NaN           NaN           NaN   
2     1950-01-03           NaN        NaN           NaN           NaN   
3     1950-01-04           NaN        NaN           NaN           NaN   
4     1950-01-05           NaN        NaN           NaN           NaN   
...          ...           ...        ...           ...           ...   
27637 2025-12-03          62.0       77.0           NaN           NaN   
27638 2025-12-04          90.0       98.0           NaN           NaN   
27639 2025-12-05          85.0       79.0           NaN           NaN   
27640 2025-12-06          95.0       84.0           NaN           NaN   
27641 2025-12-07         100.0       93.0           NaN           NaN   

      tmax_salto tmin_salto amplitud_carrasco amplitud_salto  dia_lluvioso  
0            

In [43]:
# ===== VientoCarrasco.csv =====
df_vientocarr = dataframes["VientoCarrasco"]

df_vientocarr = df_vientocarr.rename(columns={
  "Fecha " : "fecha",
  "DirVCarrasco" : "dir_viento_carrasco",
  "IntVCarrasco" : "int_viento_carrasco",
  "IntRCarrasco" : "int_racha_viento_carrasco"
})

df_vientocarr["fecha"] = pd.to_datetime(df_vientocarr["fecha"], dayfirst=True)
df_vientocarr["fecha"] = df_vientocarr["fecha"].dt.date
df_vientocarr["fecha"] = pd.to_datetime(df_vientocarr["fecha"])
df_vientocarr["dir_viento_carrasco"] = pd.to_numeric(df_vientocarr["dir_viento_carrasco"],errors="coerce")

df_vientocarr = df_vientocarr.groupby("fecha").agg({
    "dir_viento_carrasco": "mean",                  # promedio
    "int_viento_carrasco": ["mean", "max"],         # promedio y máximo
    "int_racha_viento_carrasco": ["mean", "max"]    # promedio y máximo
})
df_vientocarr.columns = [
    "dir_viento_carrasco_prom",
    "int_viento_carrasco_prom",
    "int_viento_carrasco_max",
    "int_racha_viento_carrasco_prom",
    "int_racha_viento_carrasco_max"
]

df_vientocarr.head()

df_global = pd.merge(df_global,df_vientocarr,how="left",on="fecha")

df_global.head()

,fecha,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,dia_lluvioso,dir_viento_carrasco_prom,int_viento_carrasco_prom,int_viento_carrasco_max,int_racha_viento_carrasco_prom,int_racha_viento_carrasco_max
0,1950-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
1,1950-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
2,1950-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
3,1950-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
4,1950-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN


In [44]:
# ===== VientoSalto.csv =====
df_vientosalto = dataframes["VientoSalto"]

df_vientosalto = df_vientosalto.rename(columns={
  "Fecha/hora" : "fecha",
  "DirVSalto" : "dir_viento_salto",
  "IntVSalto" : "int_viento_salto",
  "IntRSalto" : "int_racha_viento_salto"
})

df_vientosalto["fecha"] = pd.to_datetime(df_vientosalto["fecha"], dayfirst=True)
df_vientosalto["fecha"] = df_vientosalto["fecha"].dt.date
df_vientosalto["fecha"] = pd.to_datetime(df_vientosalto["fecha"])
df_vientosalto["dir_viento_salto"] = pd.to_numeric(df_vientosalto["dir_viento_salto"],errors="coerce")

df_vientosalto = df_vientosalto.groupby("fecha").agg({
    "dir_viento_salto": "mean",                  # promedio
    "int_viento_salto": ["mean", "max"],         # promedio y máximo
    "int_racha_viento_salto": ["mean", "max"]    # promedio y máximo
})
df_vientosalto.columns = [
    "dir_viento_salto_prom",
    "int_viento_salto_prom",
    "int_viento_salto_max",
    "int_racha_viento_salto_prom",
    "int_racha_viento_salto_max"
]

df_vientosalto.head()

df_global = pd.merge(df_global,df_vientosalto,how="left",on="fecha")

df_global.head()

,fecha,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,dia_lluvioso,dir_viento_carrasco_prom,int_viento_carrasco_prom,int_viento_carrasco_max,int_racha_viento_carrasco_prom,int_racha_viento_carrasco_max,dir_viento_salto_prom,int_viento_salto_prom,int_viento_salto_max,int_racha_viento_salto_prom,int_racha_viento_salto_max
0,1950-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1950-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1950-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1950-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
# ===== Visibilidad.csv =====

df_vis = dataframes["Visibilidad"]

df_vis = df_vis.rename(columns={
    "Fecha/hora": "fecha",
    "VisibilidadCarrasco ": "vis_carrasco",
    "VisibilidadSalto ": "vis_salto"
})

df_vis["fecha"] = pd.to_datetime(
    df_vis["fecha"],
    dayfirst=True,
    infer_datetime_format=True,  # infiere si hay hora o no
    errors = "coerce"
)
df_vis["fecha"] = df_vis["fecha"].dt.date
df_vis["fecha"] = pd.to_datetime(df_vis["fecha"])

df_vis["vis_carrasco"] = pd.to_numeric(df_vis["vis_carrasco"], errors="coerce")
df_vis["vis_salto"] = pd.to_numeric(df_vis["vis_salto"], errors="coerce")

df_vis = df_vis.groupby("fecha").agg({
    "vis_carrasco": ["mean", "max", "min"],
    "vis_salto": ["mean", "max", "min"]
})

df_vis.columns = [
    "vis_carrasco_prom", "vis_carrasco_max", "vis_carrasco_min",
    "vis_salto_prom", "vis_salto_max", "vis_salto_min"
]

df_global = pd.merge(df_global,df_vis,how="left",on="fecha")

df_global

/tmp/ipykernel_69360/437028442.py:11: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_vis["fecha"] = pd.to_datetime(


,fecha,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,dia_lluvioso,...,int_viento_salto_prom,int_viento_salto_max,int_racha_viento_salto_prom,int_racha_viento_salto_max,vis_carrasco_prom,vis_carrasco_max,vis_carrasco_min,vis_salto_prom,vis_salto_max,vis_salto_min
0,1950-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1950-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1950-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1950-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27637,2025-12-03,62.0,77.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27638,2025-12-04,90.0,98.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27639,2025-12-05,85.0,79.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27640,2025-12-06,95.0,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
# ===== Fenomenos.csv =====

df_fen = dataframes["Fenomenos"]

df_fen = df_fen.rename(columns={
    "Fecha/hora": "fecha",
    "FenomenosCarrasco ": "fen_carrasco",
    "FenomenosSalto ": "fen_salto"
})
df_fen = df_fen.drop(axis=1,columns=["Unnamed: 3","Unnamed: 4"])

df_fen["fecha"] = df_fen["fecha"].astype(str).str.split(" ").str[0]
df_fen["fecha"] = pd.to_datetime(df_fen["fecha"], dayfirst=True, errors="coerce")

df_carr = pd.get_dummies(df_fen[["fecha", "fen_carrasco"]], columns=["fen_carrasco"])
df_carr_daily = df_carr.groupby("fecha").sum().clip(upper=1).reset_index()  # 0/1
df_salto = pd.get_dummies(df_fen[["fecha", "fen_salto"]], columns=["fen_salto"])
df_salto_daily = df_salto.groupby("fecha").sum().clip(upper=1).reset_index()  # 0/1
df_fen = pd.merge(df_carr_daily, df_salto_daily, on="fecha", how="outer")

#df_fen["fecha"] = pd.to_datetime(df_fen["fecha"])

df_global = pd.merge(df_global,df_fen,how="left",on="fecha")

df_global.head()


,fecha,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,dia_lluvioso,...,fen_salto_76.0,fen_salto_80.0,fen_salto_81.0,fen_salto_85.0,fen_salto_91.0,fen_salto_92.0,fen_salto_95.0,fen_salto_96.0,fen_salto_97.0,fen_salto_99.0
0,1950-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1950-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1950-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1950-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
